In [2]:
# %pip install kagglehub

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("matthewjansen/ucf101-action-recognition")

print("Path to dataset files:", path)

/home/duyth/miniconda/envs/jupyter_notebooks/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Resuming download from 3062890496 bytes (3944737769 bytes left)...
Resuming download to /home/duyth/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/4.archive (3062890496/7007628265) bytes left.


100%|██████████| 6.53G/6.53G [20:05<00:00, 3.27MB/s]  

Extracting files...


Path to dataset files: /home/duyth/.cache/kagglehub/datasets/matthewjansen/ucf101-action-recognition/versions/4


In [4]:
import json
import csv
from pathlib import Path
from collections import OrderedDict

# Path to the dataset
base_root = Path("/home/duyth/ai_coding/TubeViT/data/")
data_root = base_root / "ucf101"
annotations_dir = base_root / "annotations"
annotations_dir.mkdir(exist_ok=True)

# First, get all unique labels from both train and val to create consistent class mapping
print("Collecting all labels...")
all_labels = set()
for csv_file in [data_root / "train.csv", data_root / "val.csv"]:
    with open(csv_file, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            all_labels.add(row['label'])


In [5]:

# Create sorted label list and mapping (consistent across train/val)
sorted_labels = sorted(list(all_labels))
label_to_index = {label: idx for idx, label in enumerate(sorted_labels)}

print(f"Found {len(sorted_labels)} unique classes")
print(f"First 10 classes: {sorted_labels[:10]}")
print(f"Last 10 classes: {sorted_labels[-10:]}")


Found 101 unique classes
First 10 classes: ['ApplyEyeMakeup', 'ApplyLipstick', 'Archery', 'BabyCrawling', 'BalanceBeam', 'BandMarching', 'BaseballPitch', 'Basketball', 'BasketballDunk', 'BenchPress']
Last 10 classes: ['TennisSwing', 'ThrowDiscus', 'TrampolineJumping', 'Typing', 'UnevenBars', 'VolleyballSpiking', 'WalkingWithDog', 'WallPushups', 'WritingOnBoard', 'YoYo']


In [6]:

# Function to create annotation JSON file from CSV
def create_annotation_json(csv_path, output_json_path, label_map):
    """
    Create UCF101 annotation JSON file from CSV.
    Format: {"Class/video.avi": class_index}
    """
    annotations = OrderedDict()
    
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Extract class and video name from clip_path
            # Format: /train/Class/video.avi or /val/Class/video.avi
            clip_path = row['clip_path'].strip()
            # Remove leading slash and split
            parts = clip_path.lstrip('/').split('/')
            if len(parts) >= 2:
                class_name = parts[1]  # e.g., "Swing"
                video_name = parts[2]  # e.g., "v_Swing_g05_c02.avi"
                # Create the key in format "Class/video.avi" (torchvision format)
                key = f"{class_name}/{video_name}"
                class_index = label_map[row['label']]
                annotations[key] = class_index
    
    # Save to JSON file
    with open(output_json_path, 'w') as f:
        json.dump(annotations, f, indent=2)
    
    print(f"Created annotation file: {output_json_path}")
    print(f"Total videos: {len(annotations)}")
    return annotations


In [7]:

# Create train annotation
print("\n" + "="*50)
print("Creating train annotation...")
train_csv = data_root / "train.csv"
train_json = annotations_dir / "ucf101_01.json"  # Standard split 01 for train
train_annotations = create_annotation_json(train_csv, train_json, label_to_index)



Creating train annotation...
Created annotation file: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01.json
Total videos: 10055


In [8]:

# Create val annotation
print("\n" + "="*50)
print("Creating val annotation...")
val_csv = data_root / "val.csv"
val_json = annotations_dir / "ucf101_01_val.json"  # Separate file for validation
val_annotations = create_annotation_json(val_csv, val_json, label_to_index)



Creating val annotation...
Created annotation file: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01_val.json
Total videos: 1673


In [9]:

# Create a combined annotation file (all videos) for reference
print("\n" + "="*50)
print("Creating combined annotation (train + val)...")
combined_annotations = {**train_annotations, **val_annotations}
combined_json = annotations_dir / "ucf101_01_combined.json"
with open(combined_json, 'w') as f:
    json.dump(combined_annotations, f, indent=2)
print(f"Created combined annotation: {combined_json}")
print(f"Total videos in combined: {len(combined_annotations)}")



Creating combined annotation (train + val)...
Created combined annotation: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01_combined.json
Total videos in combined: 11728


In [10]:

# Save class mapping for reference
class_mapping_file = annotations_dir / "class_mapping.json"
class_mapping = {
    "index_to_class": {idx: label for label, idx in label_to_index.items()},
    "class_to_index": label_to_index
}
with open(class_mapping_file, 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"Created class mapping: {class_mapping_file}")

print("\n" + "="*50)
print("Summary:")
print(f"  Dataset root: {data_root}")
print(f"  Annotations directory: {annotations_dir}")
print(f"  Train videos: {len(train_annotations)}")
print(f"  Val videos: {len(val_annotations)}")
print(f"  Total classes: {len(sorted_labels)}")
print(f"\nAnnotation files created:")
print(f"  - Train: {train_json}")
print(f"  - Val: {val_json}")
print(f"  - Combined: {combined_json}")
print(f"  - Class mapping: {class_mapping_file}")
print("\n" + "="*50)
print("Usage in train.py:")
print(f"  --dataset-root: {data_root}")
print(f"  --annotation-path: {train_json} (for training)")
print(f"  Note: For validation, use train=False with the same annotation file")
print("  OR use separate annotation file for val if needed")

Created class mapping: /home/duyth/ai_coding/TubeViT/data/annotations/class_mapping.json

Summary:
  Dataset root: /home/duyth/ai_coding/TubeViT/data/ucf101
  Annotations directory: /home/duyth/ai_coding/TubeViT/data/annotations
  Train videos: 10055
  Val videos: 1673
  Total classes: 101

Annotation files created:
  - Train: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01.json
  - Val: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01_val.json
  - Combined: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01_combined.json
  - Class mapping: /home/duyth/ai_coding/TubeViT/data/annotations/class_mapping.json

Usage in train.py:
  --dataset-root: /home/duyth/ai_coding/TubeViT/data/ucf101
  --annotation-path: /home/duyth/ai_coding/TubeViT/data/annotations/ucf101_01.json (for training)
  Note: For validation, use train=False with the same annotation file
  OR use separate annotation file for val if needed


In [ ]:
# create annotation.json